# FaceDetector Processing

This is verify using multiple roi is bad enough

---

Tahapan ini mencakup pengambilan, pemrosesan dan evaluasi untuk skenario tugas dan istirahat
- Mengambil data dari file npy
- Melakukan pre-processing pada sinyal rPPG dan PPG
- Melakukan extraksi HR dan HRV (SDNN dan RMSSD)
- Evaluasi MAE, RMSE dan PC

In [55]:
## Importing Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import neurokit2 as nk
import scipy
import os
from prettytable import PrettyTable
from dataclasses import dataclass
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import numpy as np


In [56]:
@dataclass
class SignalData:
    subject: str
    task: str
    method: str  
    raw: np.ndarray
    fs: int
    preprocessed: np.ndarray = None


## Preprocessing & Signal Extraction

### Define the processing method (Preprocess, Ekstraksi HR / HRV Metrik Pipeline)
---

In [57]:
def preprocess_rppg(rppg_signal, fs_rppg=35, fs_ppg=64):
    # Target: exactly 180 seconds at 64 Hz = 11520 samples
    target_duration = 180.0  # seconds
    target_samples = 11520  # Exact target
    
    # Calculate current duration
    current_duration = len(rppg_signal) / fs_rppg
    print(f"Original samples: {len(rppg_signal)}, Current duration: {current_duration:.2f}s")
    
    # Trim to exactly 180 seconds if longer
    if current_duration > target_duration:
        samples_to_keep = int(target_duration * fs_rppg)  # 180 * 35 = 6300 samples
        rppg_trimmed = rppg_signal[:samples_to_keep]
        print(f"Trimmed to {len(rppg_trimmed)} samples ({len(rppg_trimmed)/fs_rppg:.2f}s)")
    else:
        rppg_trimmed = rppg_signal
        print(f"No trimming needed, using {len(rppg_trimmed)} samples")
    
    # Cubic spline interpolation with EXACT target samples
    duration_trimmed = len(rppg_trimmed) / fs_rppg
    t_old = np.arange(len(rppg_trimmed)) / fs_rppg
    t_new = np.linspace(0, duration_trimmed, target_samples, endpoint=False)  # endpoint=False is KEY
    cs = scipy.interpolate.CubicSpline(t_old, rppg_trimmed)
    rppg_up = cs(t_new)
    
    print(f"After interpolation: {len(rppg_up)} samples (target: {target_samples})")
    
    # Bandpass filter at new rate
    b, a = scipy.signal.butter(3, [0.7, 2.5], btype='band', fs=fs_ppg)
    rppg_filtered = scipy.signal.filtfilt(b, a, rppg_up)
    
    # Normalize
    rppg_filtered = (rppg_filtered - np.mean(rppg_filtered)) / np.std(rppg_filtered)
    
    # Final verification
    print(f"Final output: {len(rppg_filtered)} samples")
    assert len(rppg_filtered) == target_samples, f"Length mismatch: got {len(rppg_filtered)}, expected {target_samples}"
    
    return rppg_filtered

In [58]:
def preprocess_ppg(ppg_signal, fs_ppg=64):

    # Bandpass filter the PPG signal
    b, a = scipy.signal.butter(3, [0.7, 2.5], btype='band', fs=fs_ppg)
    ppg_filtered = scipy.signal.filtfilt(b, a, ppg_signal)

    # Normalize the signal
    ppg_filtered = (ppg_filtered - np.mean(ppg_filtered)) / np.std(ppg_filtered)

    return ppg_filtered


Main Methods

---

Opening the npy files for FaceDetector

In [59]:
root_path = ""
subjects = ["s51","s52", "s53","s54","s55","s56"]
tasks = ["T1"] # Rest, Task

sample_rate_gt = 64  # Hz
sample_rate_video = 35 # Hz

all_signals = []

for subject in subjects:
    for task in tasks:
        # --- Ground truth PPG ---
        gt_file = os.path.join(root_path, subject, f"bvp_{subject}_{task}.csv")
        gt_data = pd.read_csv(gt_file, header=None).values.flatten()
        gt_signal = preprocess_ppg(gt_data, fs_ppg=sample_rate_gt)

        all_signals.append(
            SignalData(
                subject=subject,
                task=task,
                method="PPG",
                raw=gt_data,
                fs=sample_rate_gt,
                preprocessed=gt_signal
            )
        )

        # --- rPPG methods ---
        for method in ["POS", "LGI", "OMIT", "GREEN", "CHROM"]:
            # file_path = os.path.join(subject, f"Optimized_Landmark_{subject}_{task}-{method}-rppg.npy")
            file_path = os.path.join(root_path, subject, f"{subject}_{task}_{method}_rppg.npy")
            rppg_raw = np.load(file_path)

            rppg_preproc = preprocess_rppg(rppg_raw,
                                           fs_rppg=sample_rate_video,
                                           fs_ppg=sample_rate_gt)

            all_signals.append(
                SignalData(
                    subject=subject,
                    task=task,
                    method=method,
                    raw=rppg_raw,
                    fs=64,
                    preprocessed=rppg_preproc
                )
            )

# Sanity check the pipeline for the result
# Expected all methods should have length of 11520 (64 Hz * 180 sec = 11520 samples long)
summary = pd.DataFrame([{
    "Subject": sig.subject,
    "Task": sig.task,
    "Method": sig.method,
    "RawLen": len(sig.raw),
    "PreprocLen": len(sig.preprocessed)
} for sig in all_signals])

print(summary.head(20))


Original samples: 6325, Current duration: 180.71s
Trimmed to 6300 samples (180.00s)
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
Trimmed to 6300 samples (180.00s)
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
Trimmed to 6300 samples (180.00s)
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
Trimmed to 6300 samples (180.00s)
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6300, Current duration: 180.00s
No trimming needed, using 6300 samples
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
Trimmed to 6300 samples (180.00s)
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples:

In [60]:
@dataclass
class MetricData:
    subject: str
    task: str
    method: str
    hr: float
    sdnn: float
    rmssd: float


In [61]:
## Define the method to compute the metrics
## This method will takes any signal (rPPG, PPG) 
# and it's sampling rate to get the HR and HRv
def compute_metrics(signal, fs):
    ## Compute the HR
    peaks , _ = scipy.signal.find_peaks(signal, prominence=0.5)

    ## Getting the RR Interval / Jeda Waktu antar Puncak / Beat
    rr_intervals = np.diff(peaks) / fs  # Convert to unit seconds

    ## Clean the RR
    rr = np.asarray(rr_intervals, dtype=float)
    rr_intervals = rr[( rr >= 0.3 ) & (rr <= 2.0)] # Clean RR interval outside 0.3 - 2.0 seconds

    ## Calculating the HR
    hr = int(60 / np.mean(rr_intervals))

    ## Converting RR interval for milis for HRV analysis
    rr_intervals_ms = rr_intervals * 1000  # Convert to milliseconds

    ## Calculating the HRV (SDNN, RMSSD)
    ## Casting to Int for easier interpretation
    sdnn = int(np.std(rr_intervals_ms))
    rmssd = int(np.sqrt(np.mean(np.square(np.diff(rr_intervals_ms)))))

    return hr, sdnn, rmssd

In [62]:
arrays = [
    [""] + ["Rest"]*6,  # Remove the Task columns since you only have T1 data
    ["Subject", "GT", "POS", "GREEN", "LGI", "OMIT", "CHROM"]  # Only 7 columns total
]


columns = pd.MultiIndex.from_arrays(arrays)

In [63]:
all_metrics = []

for sig in all_signals:
    hr, sdnn, rmssd = compute_metrics(sig.preprocessed, 64)
    all_metrics.append(MetricData(sig.subject, sig.task, sig.method, hr, sdnn, rmssd))


In [64]:
df_metrics = pd.DataFrame([{
    "Subject": metrics.subject,
    "Task": metrics.task,
    "Method": metrics.method,
    "HR": metrics.hr,
    "SDNN": metrics.sdnn,
    "RMSSD": metrics.rmssd
} for metrics in all_metrics])


In [65]:
rows = []

for subject in subjects:
    row = [subject]
    # Rest (T1) HR
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'T1' and Method == @method")["HR"]
        row.append(val.values[0] if len(val) > 0 else None)
    # # Task (T3) HR
    # for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
    #     val = df_metrics.query("Subject == @subject and Task == 'T3' and Method == @method")["HR"]
    #     row.append(val.values[0] if len(val) > 0 else None)

    rows.append(row)

df_hr = pd.DataFrame(rows, columns=columns)
df_hr

Rest                         
  Subject   GT POS GREEN LGI OMIT CHROM
0     s51   62  62    67  62   62    63
1     s52   78  89    82  89   89    88
2     s53   64  85    82  85   85    85
3     s54   87  93    76  90   90    92
4     s55   88  93    64  89   89    91
5     s56   71  84    68  79   79    80

In [66]:
methods = ["POS", "GREEN", "LGI", "OMIT", "CHROM"]
conditions = ["Rest",]

eval_results = []

for cond in conditions:
    gt = df_hr[(cond, "GT")]  # ground truth for this condition
    for method in methods:
        row=[method]
        pred = df_hr[(cond, method)]
        
        mae = mean_absolute_error(gt, pred)
        rmse = np.sqrt(np.mean((gt - pred) ** 2))
        pc, _ = pearsonr(gt, pred)
        
        eval_results.append({
            "Condition": cond,
            "Method": method,
            "MAE": mae,
            "RMSE": rmse,
            "PC": pc
        })

eval_df_hr = pd.DataFrame(eval_results)
print("HR (satuan: BPM)")
eval_df_hr

HR (satuan: BPM)


,Condition,Method,MAE,RMSE,PC
0,Rest,POS,9.333333,11.489125,0.793155
1,Rest,GREEN,10.833333,13.360389,-0.124087
2,Rest,LGI,7.333333,10.295630,0.741077
3,Rest,OMIT,7.333333,10.295630,0.741077
4,Rest,CHROM,8.166667,10.464225,0.788436
